In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_selection import f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


RANDOM_STATE = 42
TOP_K = 8


def find_feature_file() -> Path:
    """
    Find github_features.csv whether the notebook is run
    from the project root or from the notebooks/ folder.
    """

    possible_paths = [
        Path("data/processed/github_features.csv"),
        Path("../data/processed/github_features.csv"),
        Path("/data/processed/github_features.csv"),
    ]

    for path in possible_paths:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find data/processed/github_features.csv. "
        "Run Step 4 first with: docker-compose run --rm churn-api python features.py"
    )


def load_feature_dataset() -> pd.DataFrame:
    """
    Load the feature dataset created in Step 4.
    """

    feature_path = find_feature_file()
    df = pd.read_csv(feature_path)

    print(f"Loaded file: {feature_path}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    return df


def prepare_x_y(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """
    Separate features X from target y.
    username is removed because it is only an identifier.
    churned is the target label.
    """

    if "churned" not in df.columns:
        raise ValueError("The dataset must contain a 'churned' column.")

    columns_to_drop = ["churned"]

    if "username" in df.columns:
        columns_to_drop.append("username")

    X = df.drop(columns=columns_to_drop)
    y = df["churned"]

    # Convert all features to numeric. Any invalid value becomes 0.
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)

    y = pd.to_numeric(y, errors="coerce").fillna(0).astype(int)

    if y.nunique() < 2:
        raise ValueError(
            "The target column 'churned' has only one class. "
            "You need both churned = 0 and churned = 1 users. "
            "Try collecting more GitHub users or changing the churn threshold."
        )

    return X, y


def run_filter_methods(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Method 1: Filter methods.

    This combines:
    - ANOVA F-test: checks relationship between each feature and churned
    - Variance check: penalizes features that barely change
    - Correlation check: penalizes features that are highly redundant
    """

    # ANOVA F-test
    f_scores, p_values = f_classif(X, y)

    filter_df = pd.DataFrame({
        "Feature": X.columns,
        "anova_score": np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0),
        "p_value": np.nan_to_num(p_values, nan=1.0, posinf=1.0, neginf=1.0),
    })

    # Variance check
    variances = X.var()
    filter_df["variance"] = filter_df["Feature"].map(variances)
    filter_df["low_variance"] = filter_df["variance"] <= 0.01

    # Correlation check
    corr_matrix = X.corr().abs()
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    redundant_features = set()

    anova_by_feature = filter_df.set_index("Feature")["anova_score"]

    for column in upper_triangle.columns:
        highly_correlated = upper_triangle.index[upper_triangle[column] > 0.90].tolist()

        for other_column in highly_correlated:
            # If two features are highly correlated, penalize the one with lower ANOVA score.
            if anova_by_feature[column] >= anova_by_feature[other_column]:
                redundant_features.add(other_column)
            else:
                redundant_features.add(column)

    filter_df["high_correlation_penalty"] = filter_df["Feature"].isin(redundant_features)

    # Final filter score
    filter_df["filter_score"] = filter_df["anova_score"]

    # Penalize low-variance features
    filter_df.loc[filter_df["low_variance"], "filter_score"] *= 0.25

    # Penalize highly correlated/redundant features
    filter_df.loc[filter_df["high_correlation_penalty"], "filter_score"] *= 0.50

    filter_df["Filter Rank"] = (
        filter_df["filter_score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return filter_df.sort_values("Filter Rank")


def run_rfe_method(X: pd.DataFrame, y: pd.Series, top_k: int = TOP_K) -> pd.DataFrame:
    """
    Method 2: RFE Wrapper.

    RFE trains a Logistic Regression model, removes weak features,
    and keeps the selected top features.
    """

    n_features_to_select = min(top_k, X.shape[1])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    logistic_model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    rfe = RFE(
        estimator=logistic_model,
        n_features_to_select=n_features_to_select,
    )

    rfe.fit(X_scaled, y)

    rfe_df = pd.DataFrame({
        "Feature": X.columns,
        "RFE Selected": rfe.support_,
        "RFE Rank": rfe.ranking_,
    })

    return rfe_df.sort_values(["RFE Rank", "Feature"])


def run_decision_tree_method(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Method 3: Decision Tree feature importance.

    A single tree is interpretable, but it can be unstable.
    """

    tree = DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    tree.fit(X, y)

    dt_df = pd.DataFrame({
        "Feature": X.columns,
        "dt_importance": tree.feature_importances_,
    })

    dt_df["DT Rank"] = (
        dt_df["dt_importance"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return dt_df.sort_values("DT Rank")


def run_random_forest_method(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Method 4: Random Forest feature importance.

    Random Forest is usually more stable than a single Decision Tree
    because it averages many trees.
    """

    forest = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    forest.fit(X, y)

    rf_df = pd.DataFrame({
        "Feature": X.columns,
        "rf_importance": forest.feature_importances_,
    })

    rf_df["RF Rank"] = (
        rf_df["rf_importance"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    return rf_df.sort_values("RF Rank")


def create_comparison_table(
    filter_df: pd.DataFrame,
    rfe_df: pd.DataFrame,
    dt_df: pd.DataFrame,
    rf_df: pd.DataFrame,
    top_k: int = TOP_K,
) -> pd.DataFrame:
    """
    Combine the results of all 4 methods into one comparison table.
    This table is the main Step 5 output.
    """

    comparison = (
        filter_df[["Feature", "Filter Rank"]]
        .merge(rfe_df[["Feature", "RFE Selected", "RFE Rank"]], on="Feature")
        .merge(dt_df[["Feature", "DT Rank"]], on="Feature")
        .merge(rf_df[["Feature", "RF Rank"]], on="Feature")
    )

    # Votes: each method gives one vote if it considers the feature important.
    comparison["Filter Vote"] = comparison["Filter Rank"] <= top_k
    comparison["RFE Vote"] = comparison["RFE Selected"]
    comparison["DT Vote"] = comparison["DT Rank"] <= top_k
    comparison["RF Vote"] = comparison["RF Rank"] <= top_k

    comparison["Total Votes"] = (
        comparison[["Filter Vote", "RFE Vote", "DT Vote", "RF Vote"]]
        .sum(axis=1)
    )

    comparison["Average Rank"] = comparison[
        ["Filter Rank", "RFE Rank", "DT Rank", "RF Rank"]
    ].mean(axis=1)

    def decide(row):
        if row["Total Votes"] >= 3:
            return "✅ Keep"
        elif row["Total Votes"] == 2:
            return "⚠️ Optional"
        else:
            return "❌ Drop"

    comparison["Decision"] = comparison.apply(decide, axis=1)

    comparison["RFE Selected"] = comparison["RFE Selected"].map({
        True: "✅",
        False: "❌",
    })

    comparison = comparison.sort_values(
        by=["Decision", "Total Votes", "Average Rank"],
        ascending=[True, False, True],
    )

    display_columns = [
        "Feature",
        "Filter Rank",
        "RFE Selected",
        "DT Rank",
        "RF Rank",
        "Decision",
        "Total Votes",
        "Average Rank",
    ]

    return comparison[display_columns].reset_index(drop=True)


def save_step5_results(comparison_df: pd.DataFrame) -> list[str]:
    """
    Save the Step 5 table and print the selected features
    that you should later copy into model.py.
    """

    feature_path = find_feature_file()
    processed_dir = feature_path.parent

    output_path = processed_dir / "feature_selection_comparison.csv"
    comparison_df.to_csv(output_path, index=False)

    selected_features = comparison_df.loc[
        comparison_df["Decision"] == "✅ Keep",
        "Feature",
    ].tolist()

    # Safety: if fewer than 5 features are marked Keep, add Optional features.
    if len(selected_features) < 5:
        optional_features = comparison_df.loc[
            comparison_df["Decision"] == "⚠️ Optional",
            "Feature",
        ].tolist()

        selected_features.extend(optional_features)

    selected_features = selected_features[:TOP_K]

    print(f"\nSaved comparison table to: {output_path}")

    print("\nCopy this into model.py later:\n")
    print("SELECTED_FEATURES = [")
    for feature in selected_features:
        print(f'    "{feature}",')
    print("]")

    return selected_features


def run_step5_feature_selection() -> tuple[pd.DataFrame, list[str]]:
    """
    Run all Step 5 methods and return:
    - comparison table
    - selected features
    """

    df = load_feature_dataset()
    X, y = prepare_x_y(df)

    print("\nClass balance:")
    print(y.value_counts())

    print("\nRunning Method 1: Filter methods...")
    filter_df = run_filter_methods(X, y)

    print("Running Method 2: RFE Wrapper...")
    rfe_df = run_rfe_method(X, y, top_k=TOP_K)

    print("Running Method 3: Decision Tree importance...")
    dt_df = run_decision_tree_method(X, y)

    print("Running Method 4: Random Forest importance...")
    rf_df = run_random_forest_method(X, y)

    comparison_df = create_comparison_table(
        filter_df=filter_df,
        rfe_df=rfe_df,
        dt_df=dt_df,
        rf_df=rf_df,
        top_k=TOP_K,
    )

    selected_features = save_step5_results(comparison_df)

    return comparison_df, selected_features

In [4]:
comparison_df, selected_features = run_step5_feature_selection()

comparison_df

Loaded file: ..\data\processed\github_features.csv
Rows: 19
Columns: 30

Class balance:
churned
0    18
1     1
Name: count, dtype: int64

Running Method 1: Filter methods...
Running Method 2: RFE Wrapper...
Running Method 3: Decision Tree importance...
Running Method 4: Random Forest importance...


c:\Users\CESAR\Desktop\churn predictor\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [21 22] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\CESAR\Desktop\churn predictor\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw



Saved comparison table to: ..\data\processed\feature_selection_comparison.csv

Copy this into model.py later:

SELECTED_FEATURES = [
    "days_since_last_activity",
    "account_age_days",
    "language_count",
    "has_bio",
    "has_company",
    "has_location",
    "recent_event_count",
]


,Feature,Filter Rank,RFE Selected,DT Rank,RF Rank,Decision,Total Votes,Average Rank
0,inactive_repo_ratio,11,❌,3,5,⚠️ Optional,2,5.25
1,gists_per_year,14,❌,3,7,⚠️ Optional,2,7.00
2,repo_count,9,✅,3,19,⚠️ Optional,2,8.00
3,push_event_ratio,8,❌,3,23,⚠️ Optional,2,9.25
4,avg_repo_size,17,❌,3,8,⚠️ Optional,2,10.00
5,days_since_last_repo_push,21,❌,3,6,⚠️ Optional,2,10.00
6,follower_ratio,24,❌,3,1,⚠️ Optional,2,11.75
7,days_since_last_activity,1,✅,1,2,✅ Keep,4,1.25
8,account_age_days,3,✅,3,3,✅ Keep,4,2.50
9,language_count,6,✅,3,4,✅ Keep,4,3.50
